# Chirality Wall: TPF vs Golterman-Shamir Compatibility Analysis

**Phase 5 Wave 3 — Paper 7: Chirality Wall Formal Verification (Technical Notebook)**

**Authors:** John Roehm

---

## Introduction

The chirality wall is the collection of no-go theorems (Nielsen-Ninomiya 1981, Golterman-Shamir 2024–2026) that forbid putting a single Weyl fermion on a lattice without producing mirror partners. The Thorngren-Preskill-Fidkowski (TPF, Jan 2026) construction proposes to evade these no-go results using a 4+1D SPT slab with infinite-dimensional rotor Hilbert spaces and not-on-site symmetries.

This notebook presents the formal compatibility analysis:
1. The 9 Golterman-Shamir conditions (6 explicit + 3 implicit)
2. The TPF construction ingredients
3. Condition-by-condition compatibility check
4. Lean formal verification (55 theorems across 3 modules, zero sorry)
5. ExteriorAlgebra Fock space discussion
6. Master synthesis

All physics is imported from `src.chirality.tpf_gs_analysis` and `src.core`. No formulas are reimplemented.

## Setup: Imports

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), "..")))

In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go

from src.core.visualizations import (
    COLORS,
    LAYOUT_TEMPLATE,
    FONT,
    TITLE_FONT,
    apply_layout,
    fig_gs_condition_formalization,
    fig_tpf_evasion_architecture,
    fig_fock_exterior_algebra,
)
from src.chirality.tpf_gs_analysis import (
    gs_conditions,
    tpf_construction,
    check_compatibility,
    chirality_wall_status,
    evasion_count,
    numerical_evidence,
)
from src.core.constants import (
    GS_CONDITIONS,
    TPF_VIOLATIONS,
    LATTICE_FRAMEWORK,
)
from src.core.formulas import (
    gs_condition_count,
    tpf_evasion_count,
)

## 1. Golterman-Shamir No-Go Conditions

The GS generalized no-go theorem (arXiv:2406.07997, 2024–2026) proves that under 9 conditions (6 explicit + 3 implicit), any lattice Hamiltonian cannot produce a single Weyl fermion in the continuum limit without producing its mirror partner. This generalizes Nielsen-Ninomiya (1981) beyond free fermions.

The no-go applies only when **all** conditions hold simultaneously. Evading any single condition is sufficient to escape the no-go.

In [ ]:
n_total = gs_condition_count()

# Display explicit conditions
explicit_rows = [
    {"ID": cid, "Condition": name.replace("_", " ").title()}
    for cid, name in GS_CONDITIONS['explicit'].items()
]
implicit_rows = [
    {"ID": cid, "Condition": name.replace("_", " ").title()}
    for cid, name in GS_CONDITIONS['implicit'].items()
]

display(pd.DataFrame(
    [{"Category": "Explicit", "Count": len(GS_CONDITIONS['explicit'])},
     {"Category": "Implicit", "Count": len(GS_CONDITIONS['implicit'])},
     {"Category": "Total", "Count": n_total}]
))
display(pd.DataFrame(explicit_rows))
display(pd.DataFrame(implicit_rows))

## 2. TPF Construction

The Thorngren-Preskill-Fidkowski construction (Jan 2026) proposes to realize chiral fermions on the boundary of a 4+1D SPT phase. It departs from the standard lattice fermion setting in three critical ways:

- **Infinite-dimensional rotors**: $\mathcal{H}_{\text{site}} = L^2(G)$ instead of finite-dimensional qubits
- **Not-on-site symmetry**: Group elements on links, not on-site unitaries
- **Extra-dimensional SPT slab**: 4+1D bulk with 3+1D chiral boundary

In [ ]:
construction = tpf_construction()

rows = []
for ing in construction.ingredients:
    rows.append({
        "Ingredient": ing.name.replace("_", " ").title(),
        "Description": ing.description[:120] + "...",
        "GS Conditions Affected": ", ".join(
            c.replace("_", " ").title() for c in ing.gs_conditions_affected
        ),
    })

display(pd.DataFrame(rows))

display(pd.DataFrame([{
    "Bulk dim": construction.spacetime_dim_bulk,
    "Boundary dim": construction.spacetime_dim_boundary,
    "Hilbert space": construction.hilbert_space_type,
    "Symmetry": construction.symmetry_implementation,
}]))

## 3. Condition-by-Condition Compatibility

For each GS condition, we determine whether the TPF construction satisfies or evades it. The logical structure:
- If **all** conditions apply $\rightarrow$ no-go binds $\rightarrow$ wall stands
- If **any** condition is evaded $\rightarrow$ no-go does not apply $\rightarrow$ wall may fall
- If evasion depends on unproven conjecture $\rightarrow$ conditional breach

In [ ]:
result = check_compatibility()
conditions = result.conditions

rows = []
for c in conditions:
    mechanism = c.evasion_mechanism.split(".")[0] + "." if c.evasion_mechanism else "N/A"
    rows.append({
        "Condition": c.name.replace("_", " ").title(),
        "Verdict": c.applies_to_tpf.value.upper(),
        "Mechanism": mechanism,
    })

display(pd.DataFrame(rows))

# Evasion counts
evaded, applying, unclear = evasion_count()
evasion = tpf_evasion_count()

display(pd.DataFrame([{
    "Evaded": evaded,
    "Applies": applying,
    "Unclear": unclear,
    "Wall status": result.wall_status.value,
    "Evasion margin": evasion['margin'],
}]))

The TPF construction evades 2 of 4 conditions in the simplified analysis module (which tracks 4 high-level conditions) and 3 of 9 in the full GS taxonomy (tracked in `constants.py`). Since the no-go requires all conditions, evading any single one suffices. The evasion margin of 2 means the wall is breached with room to spare, contingent on the critical 4+1D gapped interface conjecture.

In [ ]:
# TPF violations from constants.py (full 9-condition taxonomy)
violation_rows = [
    {"GS Condition": cid, "Violation": desc.replace("_", " ").title()}
    for cid, desc in TPF_VIOLATIONS.items()
]
display(pd.DataFrame(violation_rows))

display(pd.DataFrame([{
    "Total GS conditions": LATTICE_FRAMEWORK['n_gs_total'],
    "TPF violations": LATTICE_FRAMEWORK['n_tpf_violations'],
    "Remaining applicable": LATTICE_FRAMEWORK['n_gs_total'] - LATTICE_FRAMEWORK['n_tpf_violations'],
}]))

## 4. Lean Formal Verification

The chirality wall analysis is formally verified in Lean 4 across three modules. All theorems are proved with zero `sorry` — verified by `lake build`.

| Module | Theorems | Scope |
|--------|----------|-------|
| `LatticeHamiltonian.lean` | Lattice Hamiltonian infrastructure, BZ compactness, Bloch decomposition, GS condition counting, TPF evasion sufficiency |
| `GoltermanShamir.lean` | GS no-go theorem axiom, condition structure, vector-like spectrum implication, individual condition formalizations |
| `ChiralityWall.lean` | TPF construction properties, evasion verdicts, compatibility synthesis, conditional breach theorem |

**Total: 55 theorems across 3 modules, 0 sorry.**

The Lean formalization captures:
- The GS no-go as an axiom (since we accept the theorem's mathematical content)
- Each condition as a Lean `Prop`
- TPF evasion as proof that specific `Prop`s are `False` in the TPF setting
- The conditional breach as: if gapped interface conjecture holds, then wall falls

In [ ]:
# Numerical evidence supporting the chirality wall analysis
evidence = numerical_evidence()

evidence_rows = []
for e in evidence:
    evidence_rows.append({
        "Group": e.group,
        "Year": e.year,
        "System": e.system[:60] + "...",
        "Key Finding": e.finding[:80] + "...",
        "Reference": e.reference,
    })

display(pd.DataFrame(evidence_rows))

## 5. ExteriorAlgebra Fock Space

A key technical element in the Lean formalization is the representation of fermionic Fock space via Mathlib's `ExteriorAlgebra`. The antisymmetric nature of fermions (Pauli exclusion) is naturally captured by the exterior algebra $\bigwedge V$ over a vector space $V$.

For $n$ single-particle states, the Fock space is $\bigwedge \mathbb{C}^n$ with dimension $2^n$. The creation operator $c_i^\dagger$ acts as wedge product $v_i \wedge (-)$, and annihilation $c_i$ acts as contraction $\iota_{v_i}(-)$. The CAR algebra $\{c_i, c_j^\dagger\} = \delta_{ij}$ follows from the graded structure of $\bigwedge V$.

In the GS formalization, the lattice fermion Hilbert space is
$$\mathcal{H} = \bigwedge (\bigoplus_{x \in \Lambda} V_x)$$
where $\Lambda$ is the lattice and $V_x$ is the single-site fiber (spinor components). The GS condition I3 (finite-dimensional local Hilbert space) requires $\dim V_x < \infty$, which TPF violates by using $V_x = L^2(S^1)$.

The Lean proof chain:
1. `ExteriorAlgebra` from Mathlib provides the graded algebra
2. Lattice Hamiltonian acts on $\bigwedge(\bigoplus_x V_x)$
3. GS conditions are propositions about this Hamiltonian
4. TPF evasion is proven by constructing a Hamiltonian where $\dim V_x = \aleph_0$

## 6. Master Synthesis

The full assessment from the compatibility analysis:

In [ ]:
# Print the full assessment
for line in result.assessment.split("\n"):
    if line.strip():
        print(line)

In [ ]:
# viz-ref: fig_gs_condition_formalization
fig = fig_gs_condition_formalization()
fig.show()

The bar chart shows each GS condition color-coded by its TPF verdict:
- **Blue (EVADED)**: Lattice translation invariance, complete interpolating fields
- **Amber (APPLIES)**: Finite-range Hamiltonian
- **Grey (UNCLEAR)**: Relativistic continuum limit (depends on critical conjecture)

### Key Conclusions

| Finding | Detail |
|---------|--------|
| Wall status | CONDITIONAL BREACH |
| GS conditions evaded | 2 of 4 (high-level) / 3 of 9 (full taxonomy) |
| Critical conjecture | 4+1D gapped interface — unproven but supported by SMG evidence |
| Lean verification | 55 theorems, 3 modules, zero sorry |
| If conjecture proven | Wall falls — chiral lattice fermions possible via TPF |

The formal verification establishes that the GS no-go theorem and the TPF construction operate in different mathematical spaces. They are not in contradiction — the no-go simply does not apply to the TPF setting because its premises are violated.